# Pipeline Walkthrough Notebook

This is a detailed notebook that runs through this projects whole pipeline using our Promping Strategy 5 (json formatting w/ repair loop integration).  
  
Pipeline:  
Prompt --> Repair loop --> Validator --> Dataset Entry




Step 1: Create a prompt.  

Our Prompt at our last level (json structured promping w/ repair loop) is going to be:  
  
"Return valid JSON only. Do not include Markdown. The JSON schema must be: {"description": string, "framework": "cirq", "num_qubits": integer, "code": string, "expected_gates": list[string], "measurement_included": boolean, "expected_behavior": string} User request: Prepare the Bell state |Phi+> on two qubits."  
  
Step 2: Create the Control Circuit.

This Prompt should return the expected code:

In [2]:
import cirq # Cirq is a Python Library for writing, manipulating, and optimizing quantum circuits and running them against quantum computers and simulators.

q0,q1 = cirq.LineQubit.range(2) # Create two qubits

circuit = cirq.Circuit() # Create an empty quantum circuit
circuit += cirq.H(q0) # Apply Hadamard gate to the first qubit. Puts q0 into superposition.
circuit += cirq.CNOT(q0, q1) # Apply CNOT gate with q0 as control and q1 as target. Entangles q0 and q1.

print(f"Bell Circuit:\n{circuit}") # Print the circuit

Bell Circuit:
0: ───H───@───
          │
1: ───────X───


Now that we have our prompt prepared and our controll circuit, we have to bring it to our repair loop but in order to do that we need to go in depth on how our devices work. 

## The Validator layout 

This next code block goes over the Validator and how each piece works in action. While the start of our Promping Strategy 5 feeds into the Repair Loop. The Validator is absolutely vital to the Repair loop and will get further integrated through this walkthrough. 


In [6]:
import ast
from collections import Counter

import cirq
import numpy as np

# Helper functions for analyzing circuits

def operations(circuit): # Returns a list of all operations in the circuit
    return [op for moment in circuit for op in moment.operations]


def gate_counts(circuit): # Returns the number of each type of gate in the circuit, excluding measurements
    return Counter(
        repr(op.gate)
        for op in operations(circuit)
        if not isinstance(op.gate, cirq.MeasurementGate)
    )


def has_measurement(circuit): # Checks if the circuit contains any measurement operations
    return any(
        isinstance(op.gate, cirq.MeasurementGate)
        for op in operations(circuit)
    )


def has_simulation_call(tree): # Checks if the code contains any calls to simulation methods (simulate, run, run_sweep, sample)
    simulation_methods = {"simulate", "run", "run_sweep", "sample"}

    return any(
        isinstance(node, ast.Call)
        and isinstance(node.func, ast.Attribute)
        and node.func.attr in simulation_methods
        for node in ast.walk(tree)
    )


def statevector(circuit): # Returns the final state vector of the circuit after removing terminal measurements
    if not circuit.are_all_measurements_terminal():
        raise ValueError("Measurements must be terminal")

    circuit = cirq.drop_terminal_measurements(circuit)
    qubits = sorted(circuit.all_qubits())
    return cirq.final_state_vector(circuit, qubit_order=qubits)


def validate_circuit( # The Heart of the validation logic. This function checks the provided LLM code against a control circuit and expected properties.
    llm_code,
    control_circuit,
    expected_num_qubits, 
    measurement_expected=False,
    simulation_expected=False,
):
    checks = {  # These are the checks that will be performed on the LLM code. Each check corresponds to a specific aspect of the code or circuit.
        "syntax": False,
        "execution": False,
        "qubits": False,
        "gates": False,
        "measurement": False,
        "statevector": False,
        "simulation": False,
    }
    errors = []

    variables = {}

    try:
        exec(llm_code, variables) # Execute the LLM code in a controlled environment and capture any exceptions that occur during execution. This allows us to check for syntax errors and runtime errors.
        checks["syntax"] = True
        checks["execution"] = True
    except SyntaxError as exc: # If the code has a syntax error, we catch it and record the error.
        errors.append("invalid_syntax")
        return finish(checks, errors, str(exc))
    except Exception as exc: # If the code has a runtime error, we catch it and record the error.
        checks["syntax"] = True
        errors.append("execution_error")
        return finish(checks, errors, str(exc))


    # Check whether simulation appears in the code
    code_lower = llm_code.lower()
    simulation_found = (
        ".simulate(" in code_lower
        or ".run(" in code_lower
    )

    checks["simulation"] = simulation_found == simulation_expected

    if not checks["simulation"]:
        if simulation_expected:
            errors.append("missing_simulation")
        else:
            errors.append("unexpected_simulation")


    # Get the circuit created by the code
    circuit = variables.get("circuit")

    if not isinstance(circuit, cirq.Circuit): # If the code did not define a Cirq circuit named 'circuit', we record an error and return the results.
        errors.append("no_circuit_found")
        return finish(
            checks,
            errors,
            "The code did not define a Cirq circuit named 'circuit'.",
        )

    checks["qubits"] = len(circuit.all_qubits()) == expected_num_qubits # Checks if the number of qubits in the circuit matches the expected number of qubits.
    if not checks["qubits"]:
        errors.append("wrong_qubit_count")

    checks["gates"] = gate_counts(circuit) == gate_counts(control_circuit) # Checks if the gates in the circuit match the gates in the control circuit.
    if not checks["gates"]:
        errors.append("wrong_gates")

    measurement_found = has_measurement(circuit)
    checks["measurement"] = measurement_found == measurement_expected # Checkks if the presence of measurement operations in the circuit matches the expected presence of measurements.
    if not checks["measurement"]:
        errors.append(
            "missing_measurement"
            if measurement_expected
            else "unexpected_measurement"
        )

    try: # If the circuit doesn't have terminal measurements, we can compute the state vector and compare it to the expected state vector from the control circuit.
        actual = statevector(circuit)
        expected = statevector(control_circuit)
        print("actual", actual)
        print("expected", expected)

        checks["statevector"] = (
            actual.shape == expected.shape
            and cirq.equal_up_to_global_phase(actual, expected)
        )
    except Exception:
        checks["statevector"] = False

    if not checks["statevector"]:
        errors.append("wrong_statevector")

    return finish(checks, errors)


def finish(checks, errors, reason=""): # This function calculates the final score and verdict based on the checks performed and any errors encountered. It returns a dictionary containing the results of the validation.
    passed = sum(checks.values())
    total = len(checks)

    return {
        "checks": checks,
        "error_categories": errors,
        "score": round(100 * passed / total),
        "fully_correct": passed == total,
        "verdict": "success" if not errors else errors[0],
        "reason": reason,
    }

Now that we better understand the main concepts within our Validator, we can look into how it gets integrated into our Repair Loop sequence. 

## The Repair Loop

The Repair Loop is the heart of our Prompting Strategy 5 approach. It allows for our model (Qwen3-8B) to create our prompted circuit, then make a new prompt off of the errors in that circuit to make a more correct circuit. Strategy 5 was our highest yeilding strategy with a Mean Score of 90.4, which was nearly 15 points higher than our second leading strategy.

In [7]:
import json
import os
import cirq
from openai import OpenAI


def call_llm(prompt: str) -> str: # This function calls the LLM (Language Model) with the provided prompt and returns the model's response. It uses the OpenAI API to interact with the model.
    client = OpenAI(
        api_key=os.environ["HF_TOKEN"],
        base_url="https://router.huggingface.co/v1",
    )

    completion = client.chat.completions.create(
        model="Qwen/Qwen3-8B",
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=15000,
    )

    content = completion.choices[0].message.content

    if content is None:
        raise RuntimeError("The model returned no content.")

    return content

# This json schema defines the expected structure of the output from the LLM.

JSON_SCHEMA_INSTRUCTIONS = """ 
Return valid JSON only. Do not include Markdown or explanations.

The JSON schema must be:
{
  "description": string,
  "framework": "cirq",
  "num_qubits": integer,
  "code": string,
  "expected_gates": list[string],
  "measurement_included": boolean,
  "expected_behavior": string
}
"""

# This is a dictonary that maps error categories to human-readable error messages. It is used to provide feedback to the modek when their code fails validation.

ERROR_MESSAGES = {
    "invalid_code": (
        "Return valid JSON containing a 'code' field."
    ),
    "invalid_syntax": (
        "The Python syntax is invalid. Return valid executable Python."
    ),
    "execution_error": (
        "The Python code caused an error when executed."
    ),
    "no_circuit_found": (
        "Define a variable named 'circuit' containing a cirq.Circuit."
    ),
    "wrong_qubit_count": (
        "Use the required number of qubits."
    ),
    "wrong_gates": (
        "Use the correct gates and the correct number of each gate."
    ),
    "missing_measurement": (
        "Add a measurement to the circuit."
    ),
    "unexpected_measurement": (
        "Do not add measurements to the circuit."
    ),
    "missing_simulation": (
        "Create a Cirq simulator and simulate or run the circuit."
    ),
    "unexpected_simulation": (
        "Do not simulate or run the circuit."
    ),
    "wrong_statevector": (
        "The circuit produces the wrong final state. Correct the gate sequence."
    ),
}


def build_initial_prompt(user_request: str) -> str: # This function builds the first prompt to send to the LLM. It includes the JSON schema instructions and the user's request.
    return (
        f"{JSON_SCHEMA_INSTRUCTIONS}\n"
        f"User request:\n{user_request}"
    )


def build_repair_prompt( # This function builds a prompt to send to the LLM when the previous code failed validation. It includes the previous user request, the code that failed validation, and the validation results in order to help the model understand where it went wrong
    original_prompt,
    bad_code,
    validation_result,
):
    rules = [
        ERROR_MESSAGES[error]
        for error in validation_result["error_categories"]
        if error in ERROR_MESSAGES
    ]

    rules_text = "\n".join(
        f"- {rule}" for rule in rules
    )

    return f"""
{JSON_SCHEMA_INSTRUCTIONS}

The previous code failed validation.

Original user request: 
{original_prompt}

Failure:
{validation_result["verdict"]}

Reason:
{validation_result["reason"]}

Issues to fix:
{rules_text}

Previous code:
{bad_code}

Return corrected JSON only.
"""


def parse_model_output(raw_output): # This function takes the model output and turns it into a runnable object.  
    text = raw_output.strip()

    if text.startswith("```json"):
        text = text.removeprefix("```json")
        text = text.removesuffix("```").strip()

    elif text.startswith("```"):
        text = text.removeprefix("```")
        text = text.removesuffix("```").strip()

    return json.loads(text)


def run_repair_loop( # This function brings everything together and through a few user inputs it innitiates the repair loop cycle. 
    original_prompt,
    control_circuit,
    simulation_expected=False,
    max_attempts=10,
):
    current_prompt = build_initial_prompt(original_prompt) # Creates the current prompt.

    expected_num_qubits = len( # Gets qubit count from control circuit.
        control_circuit.all_qubits()
    )

    measurement_expected = any( # Checks for measurement gates.
        isinstance(op.gate, cirq.MeasurementGate)
        for op in control_circuit.all_operations()
    )

    for attempt in range(max_attempts): # THIS IS THE FOR LOOP. 
        print(
            f"\n--- Attempt {attempt + 1} " # Each attempt starts with "Attempt _/max_attempts". For our project we had a max of 10 attempts.
            f"of {max_attempts} ---"
        )

        raw_output = call_llm(current_prompt) # Gets EVERYTHING the LLM generated.
        print(raw_output)

        try:
            payload = parse_model_output(raw_output) # Parses the models output (takes the code out of it).
            code = payload["code"]

        except (json.JSONDecodeError, KeyError, TypeError) as exc: # Breaks if code isn't a json object.
            validation_result = {
                "fully_correct": False,
                "verdict": "invalid_code",
                "reason": (
                    "The response was not valid JSON "
                    f"with a 'code' field: {exc}"
                ),
                "error_categories": ["invalid_code"],
            }

            bad_code = raw_output # Sets the output as bad_code for next attempt.

        else:
            validation_result = validate_circuit( # If code is properly structured, it runs it through the validator.
                llm_code=code,
                control_circuit=control_circuit,
                expected_num_qubits=expected_num_qubits,
                measurement_expected=measurement_expected,
                simulation_expected=simulation_expected,
            )

            bad_code = code # Sets the code as bad_code for next attempt.

        if validation_result["fully_correct"]: # If the validation is fully correct it breaks and returns the code. 
            print("\nValidation passed!")
            print(
                "Score:",
                validation_result["score"],
            ) 
            return bad_code

        print( # If it isnt fully correct but still passes it returns the error categories it failed.
            "Validation failed:",
            validation_result["error_categories"],
        )

        current_prompt = build_repair_prompt( # Reassigns current prompt as a new prompt built with the build_repair_prompt function. It then starts the loop from the top with a this new prompt. 
            original_prompt,
            bad_code,
            validation_result,
        )

    raise RuntimeError(
        f"The model failed after {max_attempts} attempts." # If the model doesn't generate the correct circuit after a assigned amount of attempts, the loop stops and raises a RuntimeError. 
    )

Now that you understand how both the Validator and Repair Loop work, we can now call the repair loop to build a bell circuit!

In [8]:
run_repair_loop(
    original_prompt="""Return valid JSON only. Do not include Markdown. The JSON schema must be: {"description": string, "framework": "cirq", "num_qubits": integer, "code": string, "expected_gates": list[string], "measurement_included": boolean, "expected_behavior": string} User request: Prepare the Bell state |Phi+> on two qubits.""",
    control_circuit=circuit,
    simulation_expected=False,
    max_attempts=10,   
)


--- Attempt 1 of 10 ---


{
  "description": "Prepares the Bell state |Φ+> using two qubits with a Hadamard gate followed by a CNOT gate.",
  "framework": "cirq",
  "num_qubits": 2,
  "code": "import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.H(q0),\n    cirq.CNOT(q0, q1)\n])\nprint(circuit)",
  "expected_gates": ["Hadamard", "CNOT"],
  "measurement_included": false,
  "expected_behavior": "Creates the entangled Bell state |Φ+> = (|00> + |11>)/√2 through quantum superposition and entanglement."
}
0: ───H───@───
          │
1: ───────X───
actual [0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]
expected [0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]

Validation passed!
Score: 100


'import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.H(q0),\n    cirq.CNOT(q0, q1)\n])\nprint(circuit)'

Congratulations! The repair loop got the correct circuit after its first attempt! However, to maintain a homogenous dataset we'll plug the generate code back into the Validator to get a json entry like the other strategies. 

In [ ]:
code = "import cirq\nq0, q1 = cirq.LineQubit.range(2)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.H(q0),\n    cirq.CNOT(q0, q1)\n])\nprint(circuit)" # Assigns our LLM-generated code to the variable code. 

validate_circuit(code, circuit, 2, False, False) 

0: ───H───@───
          │
1: ───────X───
actual [0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]
expected [0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


{'checks': {'syntax': True,
  'execution': True,
  'qubits': True,
  'gates': True,
  'measurement': True,
  'statevector': True,
  'simulation': True},
 'error_categories': [],
 'score': 100,
 'fully_correct': True,
 'verdict': 'success',
 'reason': ''}

This is the entry that we decided to plugin to our dataset for all 5 of our prompting strategies. 

While the bell-state circuit is very simple, I think that the beauty of the repair validator is when it has to takle more rigorous circuits. 
## One more (hard) Example
Our next example is going to be a circuit that only Strategy 5 managed to solve: Bit-flip Error Correction Code. 

Bit-flip Error Correction is very important because it protects one of the most common errors in quantum computing and helps in larger circuits as a building block to handle more errors. 

Our Prompt for Strategy 5 Bit-flip Error Correction is: 

"""Return valid JSON only. Do not include Markdown. The JSON schema must be: {"description": string, "framework": "cirq", "num_qubits": integer, "code": string, "expected_gates": list[string], "measurement_included": boolean, "expected_behavior": string} User request: Encode a single logical qubit into the three-qubit bit-flip code using two CNOT gates, apply a Pauli-X error on one physical qubit, then correct it via majority-vote syndrome measurement"""

In [25]:
q0, q1, q2 = cirq.LineQubit.range(3) # Creates three qubits.

bitflip = cirq.Circuit(
    cirq.CNOT(q0, q1), # Entangles q0 and q1 
    cirq.CNOT(q0, q2), # Entangles q0 and q2
    cirq.X(q1), # This is the bitflip this circuit is searching for.
    cirq.measure(q0, q1, q2, key='syndrome'), # Measures all three qubits
)

# sim = cirq.Simulator() # You can run this simulation to see how the circuit finds the bitflip.
# result = sim.run(bitflip, repetitions=1)
# print(result.measurements)

print(bitflip)

          ┌──┐
0: ───@────@─────M('syndrome')───
      │    │     │
1: ───X────┼X────M───────────────
           │     │
2: ────────X─────M───────────────
          └──┘


Now that we have our prompt and control circuit, lets run the repair loop and see how it does!

In [18]:
run_repair_loop(
    original_prompt= """Return valid JSON only. Do not include Markdown. The JSON schema must be: {"description": string, "framework": "cirq", "num_qubits": integer, "code": string, "expected_gates": list[string], "measurement_included": boolean, "expected_behavior": string} User request: Encode a single logical qubit into the three-qubit bit-flip code using two CNOT gates, apply a Pauli-X error on one physical qubit, then correct it via majority-vote syndrome measurement.""",
    control_circuit=bitflip,
    simulation_expected=False,
    max_attempts=10,
)


--- Attempt 1 of 10 ---


{
  "description": "Encodes a single logical qubit into the three-qubit bit-flip code using two CNOT gates, applies a Pauli-X error on one physical qubit, then corrects it via majority-vote syndrome measurement.",
  "framework": "cirq",
  "num_qubits": 3,
  "code": "q0, q1, q2 = cirq.LineQubit.range(3)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.CNOT(q0, q1),\n    cirq.CNOT(q0, q2),\n    cirq.X(q1),\n    cirq.measure(q0, key='syndrome0'),\n    cirq.measure(q1, key='syndrome1'),\n    cirq.measure(q2, key='syndrome2')\n])\nprint(circuit)",
  "expected_gates": ["CNOT", "X", "measure"],
  "measurement_included": true,
  "expected_behavior": "The syndrome measurements of the three qubits will identify the location of the Pauli-X error via majority-vote, enabling correction by flipping the corresponding physical qubit."
}
Validation failed: ['execution_error']

--- Attempt 2 of 10 ---


{
  "description": "Encode a single logical qubit into the three-qubi

"import cirq\nq0, q1, q2 = cirq.LineQubit.range(3)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.CNOT(q0, q1),\n    cirq.CNOT(q0, q2),\n    cirq.X(q1),\n    cirq.measure(q0, key='syndrome0'),\n    cirq.measure(q1, key='syndrome1'),\n    cirq.measure(q2, key='syndrome2')\n])\nprint(circuit)"

This shows how our repair loop works to fix its previous errors. Initially the model failed to import cirq, however,in the second itteration, the model fixed its error and generated the correct circuit.  
  
Once again, we will validate what the model generated to keep our dataset homogenous. 

In [24]:
validate_circuit(
    llm_code="import cirq\nq0, q1, q2 = cirq.LineQubit.range(3)\ncircuit = cirq.Circuit()\ncircuit.append([\n    cirq.CNOT(q0, q1),\n    cirq.CNOT(q0, q2),\n    cirq.X(q1),\n    cirq.measure(q0, key='syndrome0'),\n    cirq.measure(q1, key='syndrome1'),\n    cirq.measure(q2, key='syndrome2')\n])\nprint(circuit)",
    control_circuit=bitflip,
    expected_num_qubits=3, 
    measurement_expected=True,
    simulation_expected=False,
)

          ┌──┐
0: ───@────@─────M('syndrome0')───
      │    │
1: ───X────┼X────M('syndrome1')───
           │
2: ────────X─────M('syndrome2')───
          └──┘
actual [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
expected [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]


{'checks': {'syntax': True,
  'execution': True,
  'qubits': True,
  'gates': True,
  'measurement': True,
  'statevector': True,
  'simulation': True},
 'error_categories': [],
 'score': 100,
 'fully_correct': True,
 'verdict': 'success',
 'reason': ''}